# Hello LangGraph
이곳에서 LangGraph 에이전트 테스트를 진행할 수 있습니다.

In [17]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import  TypedDict
from typing import Annotated
import operator
from datetime import datetime
from typing import Literal
from typing import Union
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

llm = init_chat_model("openai:gpt-4o")

conn = sqlite3.connect("memory.db", check_same_thread=False,)

In [18]:
class State(MessagesState):
    custom_stuff : str


graph_builder = StateGraph(State)


In [19]:
@tool
def get_weather(city : str) -> str:
    """Get the current weather for a given city."""
    return f"The current weather in {city} is sunny with a temperature of 25°C."

llm_with_tools = llm.bind_tools([get_weather])

def chatbot(state : State):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages" : [response]
    }

tool_node = ToolNode(
    tools=[
        get_weather
    ]
)

In [ ]:


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

graph.invoke(
    {
        "messages" : [
            {"role":"user", "content":"서울, 부산, 대구 날씨는 어때?"}
        ]
    },
    config={
        "configurable" : {
            "thread_id" : "2"
        },
        "recursion_limit" : 5
    }
)


{'messages': [HumanMessage(content='서울, 부산, 대구 날씨는 어때?', additional_kwargs={}, response_metadata={}, id='05dd8127-b6e8-4366-8e70-1783bdff40d4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 58, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d0d30e3ff1', 'id': 'chatcmpl-DP8gBTXQulmQPZFwyjBVZMQgL6zU8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3f53-944c-7012-8b02-91c18f5a021d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seoul'}, 'id': 'call_0aSshKMJbFmN7I08PmXGRozw', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Busan'}, 'id': 'call_yd5At6sKNvngIFWcVHR

In [22]:
for state in graph.get_state_history({       "configurable" : {
            "thread_id" : "2"
        },}):
    print(state.next)

()
('chatbot',)
('__start__',)
('chatbot',)
('tools',)
('chatbot',)
('__start__',)
